In [1]:
import pandas as pd

In [3]:
file_path = '/home/mohnish/amazon_sentiment_analysis/data/Reviews.csv'
columns_to_keep = [
    'Score', 
    'HelpfulnessNumerator', 
    'HelpfulnessDenominator', 
    'Text'
    ]
df = pd.read_csv(file_path, nrows = 100000, usecols = columns_to_keep)
print(f"Dataset Shape: {df.shape}")

Dataset Shape: (100000, 4)


In [4]:

print("Missing values per column:")
print(df.isnull().sum())

df.dropna(subset=['Text'], inplace=True)

Missing values per column:
HelpfulnessNumerator      0
HelpfulnessDenominator    0
Score                     0
Text                      0
dtype: int64


In [5]:
invalid_mask = df['HelpfulnessNumerator'] > df['HelpfulnessDenominator']
print(f"Removing {invalid_mask.sum()} rows with impossible helpfulness scores.")

df = df[df['HelpfulnessNumerator'] <= df['HelpfulnessDenominator']]

Removing 2 rows with impossible helpfulness scores.


In [6]:
print("\nReview Score Distribution:")
print(df['Score'].value_counts().sort_index())


Review Score Distribution:
Score
1     9318
2     5568
3     8059
4    14642
5    62411
Name: count, dtype: int64


In [7]:
df = df[df['Score'] != 3]

df['Sentiment'] = df['Score'].apply(lambda x: 1 if x > 3 else 0)

print(f"Final shape after validation: {df.shape}")

Final shape after validation: (91939, 5)


In [ ]:
import re

def clean_text(text):
    
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()
    
    return text

df['Cleaned_Text'] = df['Text'].apply(clean_text)

print("Comparison of Raw vs Cleaned:")
display(df[['Text', 'Cleaned_Text']].head(3))

Comparison of Raw vs Cleaned:


,Text,Cleaned_Text
0,I have bought several of the Vitality canned d...,i have bought several of the vitality canned d...
1,Product arrived labeled as Jumbo Salted Peanut...,product arrived labeled as jumbo salted peanut...
2,This is a confection that has been around a fe...,this is a confection that has been around a fe...


In [9]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

def lemmatize_pipe(texts):
    clean_texts = []
    for doc in nlp.pipe(texts, batch_size=500):
        tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_space]
        clean_texts.append(" ".join(tokens))
    return clean_texts

df['Final_Clean_Text'] = lemmatize_pipe(df['Cleaned_Text'])

print("Final NLP Result:")
display(df[['Text', 'Final_Clean_Text']].head(3))

Final NLP Result:


,Text,Final_Clean_Text
0,I have bought several of the Vitality canned d...,buy vitality can dog food product find good qu...
1,Product arrived labeled as Jumbo Salted Peanut...,product arrive label jumbo salt peanutsthe pea...
2,This is a confection that has been around a fe...,confection century light pillowy citrus gelati...


In [10]:
df['Reliability'] = df['HelpfulnessNumerator'] / (df['HelpfulnessDenominator'] + 1)

print(df['Reliability'].describe())

count    91939.000000
mean         0.271681
std          0.316425
min          0.000000
25%          0.000000
50%          0.000000
75%          0.500000
max          0.992895
Name: Reliability, dtype: float64


In [11]:
df['Lead_Score'] = df['Sentiment'] * df['Reliability']

top_leads = df.sort_values(by='Lead_Score', ascending=False).head(5)

print("Top 5 Positive Marketing Leads:")
display(top_leads[['Final_Clean_Text', 'Lead_Score']])

Top 5 Positive Marketing Leads:


,Final_Clean_Text,Lead_Score
96104,ecobrew reusable keurig kcup great brew coffee...,0.992895
21837,toy take effort food buster cube like occupy d...,0.992647
88407,good opportunity sample timothy kcup selection...,0.992593
4251,usually gluten free stuff scratch love day fee...,0.991150
36478,sure long miracle noodle discover sort acciden...,0.990050


In [12]:
df.to_csv('/home/mohnish/amazon_sentiment_analysis/data/Cleaned_Amazon_Reviews_Sample.csv', index=False)
print("Data Preprocessing Complete. File saved as 'Cleaned_Amazon_Reviews_Sample.csv'")

Data Preprocessing Complete. File saved as 'Cleaned_Amazon_Reviews_Sample.csv'
